In [1]:
# notebooks/02_EDA.ipynb — CELL 1

import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

os.makedirs('../plots', exist_ok=True)

# -----------------------------------
# LOAD DATASET
# -----------------------------------
df = pd.read_csv('../data/fd_loans.csv')

# -----------------------------------
# CREATE FIGURE
# -----------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

fig.suptitle(
    'FD-Backed Loan Risk Analysis — EDA',
    fontsize=14,
    fontweight='bold'
)

# -----------------------------------
# LTV RISK BANDS
# -----------------------------------
df['ltv_band'] = pd.cut(
    df['ltv'],
    bins=[0, 0.60, 0.70, 0.80, 0.90, 1.0],
    labels=['<60%', '60-70%', '70-80%', '80-90%', '90%+']
)

ltv_def = (
    df.groupby('ltv_band', observed=True)['defaulted']
    .mean() * 100
)

colors_l = [
    '#16A34A',
    '#65A30D',
    '#EAB308',
    '#F97316',
    '#EF4444'
]

bars = axes[0, 0].bar(
    ltv_def.index,
    ltv_def.values,
    color=colors_l,
    edgecolor='black',
    alpha=0.85
)

axes[0, 0].set_title(
    'Default Rate by LTV Band',
    fontweight='bold'
)

axes[0, 0].set_ylabel('Default Rate (%)')

for bar, val in zip(bars, ltv_def.values):
    axes[0, 0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.05,
        f'{val:.2f}%',
        ha='center',
        fontweight='bold',
        fontsize=9
    )

# -----------------------------------
# LTV DISTRIBUTION
# -----------------------------------
axes[0, 1].hist(
    df['ltv'],
    bins=50,
    color='#1D4ED8',
    edgecolor='white',
    alpha=0.85
)

axes[0, 1].axvline(
    0.80,
    color='red',
    linestyle='--',
    linewidth=2,
    label='80% threshold'
)

axes[0, 1].axvline(
    df['ltv'].mean(),
    color='gold',
    linewidth=2,
    label=f"Mean: {df['ltv'].mean():.2f}"
)

axes[0, 1].set_title(
    'LTV Distribution',
    fontweight='bold'
)

axes[0, 1].legend(fontsize=8)

# -----------------------------------
# CIBIL RISK ANALYSIS
# -----------------------------------
df['cibil_grp'] = pd.cut(
    df['cibil'],
    bins=[300, 600, 650, 700, 750, 900],
    labels=['<600', '600-650', '650-700', '700-750', '750+']
)

cibil_def = (
    df.groupby('cibil_grp', observed=True)['defaulted']
    .mean() * 100
)

axes[0, 2].bar(
    cibil_def.index,
    cibil_def.values,
    color=[
        '#EF4444',
        '#F97316',
        '#EAB308',
        '#65A30D',
        '#16A34A'
    ],
    edgecolor='black',
    alpha=0.85
)

axes[0, 2].set_title(
    'Default Rate by CIBIL Band',
    fontweight='bold'
)

axes[0, 2].set_ylabel('Default Rate (%)')

for i, (idx, val) in enumerate(cibil_def.items()):
    axes[0, 2].text(
        i,
        val + 0.05,
        f'{val:.2f}%',
        ha='center',
        fontweight='bold',
        fontsize=9
    )

# -----------------------------------
# FD AMOUNT VS LOAN AMOUNT
# -----------------------------------
sample = df.sample(3000)

axes[1, 0].scatter(
    sample['fd_amount'] / 1e5,
    sample['loan_amt'] / 1e5,
    c=sample['ltv'],
    cmap='RdYlGn_r',
    alpha=0.4,
    s=12
)

axes[1, 0].set_title(
    'FD Amount vs Loan Amount (color=LTV)',
    fontweight='bold'
)

axes[1, 0].set_xlabel('FD Amount (Rs lakh)')
axes[1, 0].set_ylabel('Loan Amount (Rs lakh)')

# -----------------------------------
# EMPLOYMENT RISK
# -----------------------------------
emp_def = (
    df.groupby('employment')['defaulted']
    .mean() * 100
)

axes[1, 1].bar(
    emp_def.index,
    emp_def.values,
    color=[
        '#1D4ED8',
        '#7C3AED',
        '#16A34A',
        '#EF4444'
    ],
    edgecolor='black',
    alpha=0.85
)

axes[1, 1].set_title(
    'Default Rate by Employment',
    fontweight='bold'
)

axes[1, 1].set_ylabel('Default Rate (%)')

for i, (idx, val) in enumerate(emp_def.items()):
    axes[1, 1].text(
        i,
        val + 0.05,
        f'{val:.2f}%',
        ha='center',
        fontweight='bold',
        fontsize=9
    )

# -----------------------------------
# EMI / INCOME ANALYSIS
# -----------------------------------
axes[1, 2].hist(
    df[df['defaulted'] == 0]['emi_income'].clip(0, 0.7),
    bins=50,
    alpha=0.6,
    color='green',
    label='No Default',
    density=True
)

axes[1, 2].hist(
    df[df['defaulted'] == 1]['emi_income'].clip(0, 0.7),
    bins=50,
    alpha=0.6,
    color='red',
    label='Default',
    density=True
)

axes[1, 2].axvline(
    0.35,
    color='black',
    linestyle='--',
    linewidth=2,
    label='35% threshold'
)

axes[1, 2].set_title(
    'EMI/Income: Default vs No Default',
    fontweight='bold'
)

axes[1, 2].legend(fontsize=8)

# -----------------------------------
# SAVE PLOT
# -----------------------------------
plt.tight_layout()

plt.savefig(
    '../plots/fd_eda.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/fd_eda.png")

# -----------------------------------
# INTERVIEW INSIGHTS
# -----------------------------------
print("\nYOUR 3 INTERVIEW INSIGHTS:\n")

hi_ltv = df[df['ltv'] > 0.85]['defaulted'].mean() * 100
lo_ltv = df[df['ltv'] < 0.60]['defaulted'].mean() * 100

print(
    f"1. LTV > 85%: {hi_ltv:.2f}% default rate "
    f"vs {lo_ltv:.2f}% for LTV < 60% "
    f"— {hi_ltv/lo_ltv:.1f}x higher"
)

bad_c = df[df['cibil'] < 650]['defaulted'].mean() * 100
good_c = df[df['cibil'] > 750]['defaulted'].mean() * 100

print(
    f"2. CIBIL < 650: {bad_c:.2f}% default "
    f"vs {good_c:.2f}% for CIBIL > 750"
)

ret = df[df['employment'] == 'Retired']['defaulted'].mean() * 100

self_e = df[df['employment'] == 'Self-Employed']['defaulted'].mean() * 100

print(
    f"3. Retired customers: {ret:.2f}% default "
    f"vs Self-Employed: {self_e:.2f}%"
)

Saved: plots/fd_eda.png

YOUR 3 INTERVIEW INSIGHTS:

1. LTV > 85%: 15.18% default rate vs 6.45% for LTV < 60% — 2.4x higher
2. CIBIL < 650: 13.54% default vs 5.81% for CIBIL > 750
3. Retired customers: 7.32% default vs Self-Employed: 10.47%
